# Step 01: goal → seeds

The goal is matched against the experiment's linker dictionary, so the seeds have the same concept IDs as everything else. Longest exact name wins; gene symbols and abbreviations must match case; only `[query] seed_types` (disease, chemical, gene) become seeds; MeSH headings too near the top of the tree (`min_seed_depth`: "Drug", "Neoplasms") are skipped; and a concept's own name beats an old or alias name of something else.

All the example notebooks use `experiments/examples.toml`: the default experiment with tiny limits and its own pool (`corpora/examples`). Run `00-setup` for it first: `python src/00-setup/run.py --experiment experiments/examples.toml --no-index`.

In [1]:
import importlib.util
import sys
import tomllib
from pathlib import Path

import duckdb
import pyarrow as pa

SRC = Path.cwd().parent
LACUNA = SRC.parent
ROOT = LACUNA / 'data'
experiment = tomllib.loads((LACUNA / 'experiments' / 'examples.toml').read_text())


def load(step):
    """A step's run.py as a module (every step's entry point is called run.py)."""
    sys.path.insert(0, str(SRC / step))
    spec = importlib.util.spec_from_file_location(f'step_{step[:2]}', SRC / step / 'run.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


query = load('01-query')


def show(goal):
    seeds = query.seeds(goal, experiment, ROOT)
    print(repr(goal))
    for s in seeds:
        print(f"  {s['text']!r:32} -> {s['id']:16} matched {s['alias']!r} ({s['source']}, {s['name_kind']})")

## 1. One disease

"treatment" and "avenues" aren't seed types, so only the disease is a seed.

In [2]:
show('new treatment avenues for glioblastoma')

'new treatment avenues for glioblastoma'
  'glioblastoma'                   -> mesh:D005909     matched 'Glioblastoma' (mesh, term)


## 2. A drug, a gene, a long disease name

`EGFR` matches the HGNC symbol (case must match). The disease is the longest name that matches: *non-small cell lung cancer*, not just *lung cancer*.

In [3]:
show('temozolomide resistance in glioblastoma')
show('EGFR inhibitors for non-small cell lung cancer')

'temozolomide resistance in glioblastoma'
  'temozolomide'                   -> mesh:D000077204  matched 'Temozolomide' (mesh, term)
  'glioblastoma'                   -> mesh:D005909     matched 'Glioblastoma' (mesh, term)
'EGFR inhibitors for non-small cell lung cancer'
  'EGFR'                           -> ncbigene:1956    matched 'EGFR' (hgnc, symbol)
  'non-small cell lung cancer'     -> mesh:D002289     matched 'Non-Small Cell Lung Cancer' (mesh, term)


## 3. What doesn't become a seed

"drug" is a name of *Pharmaceutical Preparations*, a top-level heading, so it's too general. HGNC lists "Alzheimer disease" as an old alias name of the APP gene; the disease heading's own name wins. Lowercase "met" (the verb) isn't the MET gene.

In [4]:
show('drug repurposing for Alzheimer disease')
show('patients who met the criteria for ependymoma')

'drug repurposing for Alzheimer disease'
  'Alzheimer disease'              -> mesh:D000544     matched 'Alzheimer Disease' (mesh, eponym)


'patients who met the criteria for ependymoma'
  'ependymoma'                     -> mesh:D004806     matched 'Ependymoma' (mesh, term)


## From the command line

```
python src/01-query/run.py --experiment experiments/default.toml
python src/01-query/run.py --experiment experiments/default.toml --goal "temozolomide resistance in glioblastoma"
```
writes `runs/<experiment>/01-query/seeds.json`.